# Imports

In [3]:
import pandas as pd
from datetime import date as dt

# Generation Functions

In [5]:
def gen_daily(df, date):
    daily_tasks = df[df['frequency'] == 'daily'][['task','specified frequency']]
    daily_tasks.columns = ['task','frequency']
    return daily_tasks

In [6]:
def gen_weekly(df, date):
    day = date.strftime('%A').lower()
    weekly_tasks = df[df['specified frequency'] == day][['task','specified frequency']]
    weekly_tasks.columns = ['task','frequency']
    return weekly_tasks

In [7]:
def gen_monthly(df, date):
    day = date.strftime('%A').lower()
    monthly_tasks = df[
    (df['frequency'] == 'monthly') & (
        ((df['specified frequency'] == 'anyday') & (date.day in range(12,18)))
        | ((df['specified frequency'] == '25th') & (date.day == 25))
        | ((df['specified frequency'] == '2nd monday') & (day == 'monday') & (((date.day % 7) + 7) == date.day))
        | ((df['specified frequency'] == '1st saturday') & (day == 'saturday') & ((date.day % 7) == date.day))
    )][['task','specified frequency']]
    monthly_tasks.columns = ['task','frequency']
    return monthly_tasks

In [8]:
def gen_otherly(df, date):
    quarterly_tasks = df[(df['frequency'] == 'quarterly') & (date.month in [1,4,7,10])][['task','frequency']]
    biyearly_tasks = df[(df['frequency'] == '6 months') & (date.month in [1,7])][['task','frequency']]
    yearly_tasks = df[(df['frequency'] == 'yearly') & (date.month == 3)][['task','frequency']]
    otherly_tasks = pd.concat([quarterly_tasks, biyearly_tasks, yearly_tasks], ignore_index = True)
    return otherly_tasks

In [9]:
def gen_all(df, date):
    total_tasks = pd.concat(
        [gen_daily(df, date), gen_weekly(df, date), gen_monthly(df, date), gen_otherly(df, date)],
        ignore_index=True)
    return total_tasks

# Other Functions

In [11]:
def tell_tasks(df, date):
    tasks = gen_all(df, date)
    for i in tasks['frequency'].drop_duplicates():
        print(f"{i} tasks:\n- {'\n- '.join(list(tasks[tasks['frequency'] == i]['task']))}\n")

In [12]:
def load_data():
    df = pd.read_excel('tasks.xlsx')[['task','frequency','specified frequency']]
    df_comp = pd.read_excel('tasks_completion.xlsx')[['date','task','frequency','completed']]
    return df, df_comp

In [13]:
def new_rows(df, date):
    total_tasks = gen_all(df, date)
    blank_row = pd.DataFrame(columns=['date','task','frequency','completed'],index=range(len(total_tasks)))
    blank_row.loc[:len(total_tasks),['date','completed']] = [date.strftime('%A, %Y-%m-%d'),False]
    blank_row.loc[:len(total_tasks),'task'] = total_tasks['task']
    blank_row.loc[:len(total_tasks),'frequency'] = total_tasks['frequency']
    return blank_row

In [14]:
def det_tasks(df_comp, blank_row, date):
    blank_row = new_rows(df, date)
    times = ['morning','midday','evening','anytime']
    days = ['sunday','monday','tuesday','wednesday','thursday','friday','saturday']
    Y1 = date.strftime('%Y')
    m1 = date.strftime('%m')
    d1 = date.strftime('%d')
    for i in df_comp['date'].drop_duplicates():
        Y2 = i.split(', ')[1].split('-')[0]
        m2 = i.split(', ')[1].split('-')[1]
        d2 = i.split(', ')[1].split('-')[2]
        if (Y1,m1,d1) != (Y2,m2,d2):
            blank_row = blank_row[blank_row['frequency'].isin(times+days)]
            df_temp = df_comp[
            (~df_comp['frequency'].isin(times+days))
            & (df_comp['completed'] == False)
            & (m1 == m2)]
            df_comp = pd.concat([blank_row, df_temp, df_comp],ignore_index=True)
            break
        elif (Y1,m1) != (Y2,m2):
            df_comp = pd.concat([blank_row, df_comp],ignore_index=True)
            break
        else:
            break
    return df_comp

In [15]:
def completion(df_comp, date):
    today_tasks = df_comp[
    (df_comp['date'] == date.strftime('%A, %Y-%m-%d'))
    & (df_comp['completed'] == False)]
    print(today_tasks[['task','frequency']])
    
    comp_tasks_str = input("What tasks have you completed?")
    comp_tasks = comp_tasks_str.split(', ')
    for i in comp_tasks:
        df_comp.loc[
        (df_comp['date'] == date.strftime('%A, %Y-%m-%d'))
        & (df_comp['task'] == i),'completed'] = True
        
    today_tasks = df_comp[
    (df_comp['date'] == date.strftime('%A, %Y-%m-%d'))
    & (df_comp['completed'] == False)]
    print(f"You have completed:\n{comp_tasks}\n\n{today_tasks[['task','frequency']]}")

# Execution

In [17]:
# Reset tasks_completion
# pd.DataFrame(columns=['date','task','frequency','completed']).to_excel('tasks_completion.xlsx')

In [18]:
# Load in tasks
df, df_comp = load_data()

In [40]:
date = dt.today()

if df_comp.empty:
    df_comp = new_rows(df, date).copy()
else:
    df_comp = det_tasks(df_comp, new_rows(df, date), date)
# completion(df_comp, date)
# df_comp.to_excel('tasks_completion.xlsx')
df_comp['kam completed'] = df_comp['completed']
df_comp.columns = ['date', 'task', 'frequency', 'car completed', 'kam completed']
df_comp.to_excel('tasks_completion.xlsx')